Transaction records for Maven Roasters, a fictitious coffee shop operating out of three NYC locations. Dataset includes the transaction date, timestamp and location, along with product-level details.

### Business Question 

How have Maven Roasters sales trended over time?

Which days of the week tend to be busiest, and why do you think that's the case?

What times of day tend to be most popular? Does the same trend hold across all locations?

Which products are sold most and least often? Which drive the most revenue for the business?

In [0]:
%sql
USE `coffee_shop_sales`.`coffee_sales`;

### How have Maven Roasters sales trended over time?

In [0]:
%sql
WITH SALES_TREND AS (
SELECT 
       MONTH(LEFT(transaction_date,10)) AS date_num
      ,MONTHNAME(LEFT(transaction_date,10)) AS transaction_date
      ,ROUND(SUM(transaction_qty * unit_price),2) AS Total_Sales
FROM coffee_shop_sales
GROUP BY date_num 
          ,MONTHNAME(LEFT(transaction_date,10))
ORDER BY date_num
),
calculations AS (
SELECT 
    date_num
    ,transaction_date
    ,total_sales
    ,LAG(Total_Sales) OVER(ORDER BY date_num) AS Previous_Month
    ,total_sales - LAG(Total_Sales) OVER(ORDER BY date_num) AS Sales_Diff
FROM SALES_TREND
)
SELECT 
    transaction_date
    ,total_sales
    ,ROUND((total_sales - previous_month )/previous_month *100,2) AS PCT_Change
FROM calculations;



Databricks visualization. Run in Databricks to view.

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# Convert to pandas for visualization
df = _sqldf.toPandas()

# Create figure
plt.figure(figsize=(12, 6))
plt.plot(df['transaction_date'], df['total_sales'], marker='o', linewidth=2, markersize=8, color='#2E86AB')
plt.fill_between(range(len(df)), df['total_sales'], alpha=0.3, color='#2E86AB')

# Formatting
plt.title('Maven Roasters Monthly Sales Trend', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Month', fontsize=12, fontweight='bold')
plt.ylabel('Total Sales ($)', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3, linestyle='--')
plt.xticks(rotation=45, ha='right')

# Add value labels on points
for i, (month, sales) in enumerate(zip(df['transaction_date'], df['total_sales'])):
    plt.text(i, sales + 1000, f'${sales:,.0f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

# Show percentage changes
print("\nMonth-over-Month Growth:")
for idx, row in df.iterrows():
    if pd.notna(row['PCT_Change']):
        print(f"{row['transaction_date']}: {row['PCT_Change']:.2f}%")

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# Convert to pandas
df = _sqldf.toPandas()

# Create figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart for transaction counts
colors = ['#FF6B6B' if rank <= 3 else '#95E1D3' for rank in df['RANK']]
ax1.bar(df['Day_of_week'], df['Number_of_Transactions'], color=colors, edgecolor='black', linewidth=1.5)
ax1.set_title('Transactions by Day of Week', fontsize=14, fontweight='bold')
ax1.set_xlabel('Day', fontsize=12, fontweight='bold')
ax1.set_ylabel('Number of Transactions', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y', linestyle='--')

# Add value labels
for i, (day, count) in enumerate(zip(df['Day_of_week'], df['Number_of_Transactions'])):
    ax1.text(i, count + 200, f'{count:,}', ha='center', fontsize=10, fontweight='bold')

# Pie chart for percentage distribution
ax2.pie(df['Percentage'], labels=df['Day_of_week'], autopct='%1.1f%%', 
        startangle=90, colors=['#FF6B6B', '#FFB6B9', '#FEC8C8', '#F3E5AB', '#95E1D3', '#A8D8EA', '#AA96DA'])
ax2.set_title('Transaction Distribution (%)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Print insights
print(f"\nBusiest Day: {df.loc[df['RANK'] == 1, 'Day_of_week'].values[0]} with {df.loc[df['RANK'] == 1, 'Number_of_Transactions'].values[0]:,} transactions")
print(f"Slowest Day: {df.loc[df['RANK'] == 7, 'Day_of_week'].values[0]} with {df.loc[df['RANK'] == 7, 'Number_of_Transactions'].values[0]:,} transactions")

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Convert to pandas
df = _sqldf.toPandas()

# Create figure with subplots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Get busiest day per month (keep first if tied)
top_days = df[df['RNK'] == 1].groupby('MonthNum').first().reset_index()
top_days = top_days.sort_values('MonthNum')

# Top chart: Busiest day per month
colors_map = {'Mon': '#FF6B6B', 'Tue': '#4ECDC4', 'Wed': '#45B7D1', 
              'Thu': '#FFA07A', 'Fri': '#98D8C8', 'Sat': '#F7DC6F', 'Sun': '#BB8FCE'}
colors = [colors_map.get(day[:3], '#95E1D3') for day in top_days['Day_of_week']]

ax1.bar(top_days['Month_of_year'], top_days['Number_of_Transactions'], 
        color=colors, edgecolor='black', linewidth=1.5)
ax1.set_title('Busiest Day of Week by Month', fontsize=14, fontweight='bold', pad=15)
ax1.set_xlabel('Month', fontsize=12, fontweight='bold')
ax1.set_ylabel('Transactions', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y', linestyle='--')

# Add day labels on bars
for i, (month, day, count) in enumerate(zip(top_days['Month_of_year'], 
                                             top_days['Day_of_week'], 
                                             top_days['Number_of_Transactions'])):
    ax1.text(i, count + 100, day, ha='center', fontsize=10, fontweight='bold')

# Bottom chart: Heatmap-style visualization of all top 2 days
# Reshape data for better visualization
df_sorted = df.sort_values(['MonthNum', 'RNK'])

# Create grouped bar chart by month showing rank distribution
for month_num in df['MonthNum'].unique():
    month_data = df[df['MonthNum'] == month_num].sort_values('RNK')
    month_name = month_data.iloc[0]['Month_of_year']
    
    for idx, row in month_data.iterrows():
        x_pos = month_num - 1
        if row['RNK'] == 1:
            ax2.bar(x_pos - 0.2, row['Number_of_Transactions'], 0.35, 
                   color='#FF6B6B', edgecolor='black', linewidth=1.5)
            ax2.text(x_pos - 0.2, row['Number_of_Transactions'] + 50, 
                    row['Day_of_week'][:3], ha='center', fontsize=8, fontweight='bold')
        elif row['RNK'] == 2:
            ax2.bar(x_pos + 0.2, row['Number_of_Transactions'], 0.35, 
                   color='#95E1D3', edgecolor='black', linewidth=1.5)
            ax2.text(x_pos + 0.2, row['Number_of_Transactions'] + 50, 
                    row['Day_of_week'][:3], ha='center', fontsize=8, fontweight='bold')

ax2.set_title('Top 2 Busiest Days per Month', fontsize=14, fontweight='bold', pad=15)
ax2.set_xlabel('Month', fontsize=12, fontweight='bold')
ax2.set_ylabel('Transactions', fontsize=12, fontweight='bold')
ax2.set_xticks(range(len(top_days)))
ax2.set_xticklabels(top_days['Month_of_year'].values)
ax2.grid(True, alpha=0.3, axis='y', linestyle='--')

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#FF6B6B', edgecolor='black', label='1st Busiest'),
                   Patch(facecolor='#95E1D3', edgecolor='black', label='2nd Busiest')]
ax2.legend(handles=legend_elements, fontsize=10)

plt.tight_layout()
plt.show()

print("\n=== Busiest Days by Month ===")
for _, row in top_days.iterrows():
    print(f"{row['Month_of_year']}: {row['Day_of_week']} ({row['Number_of_Transactions']:,} transactions)")
import pandas as pd

# Get data from previous SQL query
df = _sqldf.toPandas()

# Create horizontal bar chart
fig, ax = plt.subplots(figsize=(12, 8))

# Create labels and colors
labels = [f"{row['Month_of_year']} - {row['Day_of_week']} (#{row['RNK']})" for idx, row in df.iterrows()]
colors = ['#1f77b4' if row['RNK'] == 1 else '#ff7f0e' for idx, row in df.iterrows()]

# Plot
ax.barh(range(len(df)), df['Number_of_Transactions'], color=colors, alpha=0.7)
ax.set_yticks(range(len(df)))
ax.set_yticklabels(labels)
ax.set_xlabel('Number of Transactions', fontsize=12)
ax.set_title('Top 2 Busiest Days per Month', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# Add value labels
for idx, (i, row) in enumerate(df.iterrows()):
    ax.text(row['Number_of_Transactions'] + 100, idx, f"{row['Number_of_Transactions']:,}", 
            va='center', fontsize=9)

# Invert y-axis to show January at top
ax.invert_yaxis()

plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# Convert to pandas
df = _sqldf.toPandas()

# Create figure with gradient colors
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Color gradient based on transaction volume
colors = plt.cm.YlOrRd(df['Total_Transactions'] / df['Total_Transactions'].max())

# Bar chart
ax1.bar(df['Time_of_Day'], df['Total_Transactions'], color=colors, edgecolor='black', linewidth=1.5)
ax1.set_title('Transaction Volume by Hour of Day', fontsize=14, fontweight='bold')
ax1.set_xlabel('Time', fontsize=12, fontweight='bold')
ax1.set_ylabel('Number of Transactions', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y', linestyle='--')
ax1.tick_params(axis='x', rotation=45)

# Add value labels on peak hours
peak_hours = df.nlargest(5, 'Total_Transactions')
for idx in peak_hours.index:
    ax1.text(idx, df.loc[idx, 'Total_Transactions'] + 300, 
             f"{df.loc[idx, 'Total_Transactions']:,}", 
             ha='center', fontsize=9, fontweight='bold')

# Line chart with area fill
ax2.plot(df['Time_of_Day'], df['Total_Transactions'], marker='o', 
         linewidth=3, markersize=6, color='#2E86AB')
ax2.fill_between(range(len(df)), df['Total_Transactions'], alpha=0.4, color='#2E86AB')
ax2.set_title('Transaction Pattern Throughout the Day', fontsize=14, fontweight='bold')
ax2.set_xlabel('Time', fontsize=12, fontweight='bold')
ax2.set_ylabel('Number of Transactions', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, linestyle='--')
ax2.tick_params(axis='x', rotation=45)

# Highlight peak period (7-10 AM)
peak_start = df[df['Hour'] == 7].index[0] if len(df[df['Hour'] == 7]) > 0 else 7
peak_end = df[df['Hour'] == 10].index[0] if len(df[df['Hour'] == 10]) > 0 else 10
ax2.axvspan(peak_start, peak_end, alpha=0.2, color='red', label='Morning Rush (7-10 AM)')
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"\nPeak Hour: {df.loc[df['Total_Transactions'].idxmax(), 'Time_of_Day']} with {df['Total_Transactions'].max():,} transactions")
print(f"Slowest Hour: {df.loc[df['Total_Transactions'].idxmin(), 'Time_of_Day']} with {df['Total_Transactions'].min():,} transactions")
import pandas as pd

# Get data from previous SQL query
df = _sqldf.toPandas()

# Create line chart
plt.figure(figsize=(12, 6))
plt.plot(df['Hour'], df['Total_Transactions'], marker='o', linewidth=2, markersize=8, color='#2ca02c')
plt.fill_between(df['Hour'], df['Total_Transactions'], alpha=0.3, color='#2ca02c')
plt.title('Transaction Volume by Hour of Day', fontsize=14, fontweight='bold')
plt.xlabel('Hour', fontsize=12)
plt.ylabel('Total Transactions', fontsize=12)
plt.xticks(df['Hour'], df['Time_of_Day'], rotation=45)
plt.grid(True, alpha=0.3)

# Highlight peak hour
peak_hour = df.loc[df['Total_Transactions'].idxmax()]
plt.axvline(x=peak_hour['Hour'], color='red', linestyle='--', alpha=0.5, label=f'Peak: {peak_hour["Time_of_Day"]}')
plt.legend()

plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# Convert to pandas
df = _sqldf.toPandas()

# Create horizontal bar chart
fig, ax = plt.subplots(figsize=(12, 6))

colors = plt.cm.Greens(range(len(df), 0, -1))
ax.barh(df['product_type'], df['Quantity_Sold'], color=colors, edgecolor='black', linewidth=1.5)

ax.set_title('Top 5 Most Sold Products', fontsize=14, fontweight='bold')
ax.set_xlabel('Quantity Sold', fontsize=12, fontweight='bold')
ax.set_ylabel('Product', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x', linestyle='--')

# Add value labels
for i, (product, qty) in enumerate(zip(df['product_type'], df['Quantity_Sold'])):
    ax.text(qty + 200, i, f'{qty:,}', va='center', fontsize=10, fontweight='bold')

ax.invert_yaxis()
plt.tight_layout()
plt.show()
import pandas as pd

# Get data from previous SQL query
df = _sqldf.toPandas()

# Create horizontal bar chart
plt.figure(figsize=(10, 6))
plt.barh(df['product_type'], df['Quantity_Sold'], color='#1f77b4', alpha=0.8)
plt.title('Top 5 Most Sold Products', fontsize=14, fontweight='bold')
plt.xlabel('Quantity Sold (Transactions)', fontsize=12)
plt.ylabel('Product Type', fontsize=12)
plt.grid(True, alpha=0.3, axis='x')

# Add value labels
for idx, row in df.iterrows():
    plt.text(row['Quantity_Sold'] + 200, idx, f"{row['Quantity_Sold']:,}", va='center', fontsize=10)

plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# Convert to pandas
df = _sqldf.toPandas()

# Create horizontal bar chart
fig, ax = plt.subplots(figsize=(12, 6))

colors = plt.cm.Reds(range(len(df), 0, -1))
ax.barh(df['product_type'], df['Quantity_Sold'], color=colors, edgecolor='black', linewidth=1.5)

ax.set_title('Top 5 Least Sold Products', fontsize=14, fontweight='bold')
ax.set_xlabel('Quantity Sold', fontsize=12, fontweight='bold')
ax.set_ylabel('Product', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x', linestyle='--')

# Add value labels
for i, (product, qty) in enumerate(zip(df['product_type'], df['Quantity_Sold'])):
    ax.text(qty + 5, i, f'{qty:,}', va='center', fontsize=10, fontweight='bold')

ax.invert_yaxis()
plt.tight_layout()
plt.show()
import pandas as pd

# Get data from previous SQL query
df = _sqldf.toPandas()

# Create horizontal bar chart
plt.figure(figsize=(10, 6))
plt.barh(df['product_type'], df['Quantity_Sold'], color='#d62728', alpha=0.8)
plt.title('Top 5 Least Sold Products', fontsize=14, fontweight='bold')
plt.xlabel('Quantity Sold (Transactions)', fontsize=12)
plt.ylabel('Product Type', fontsize=12)
plt.grid(True, alpha=0.3, axis='x')

# Add value labels
for idx, row in df.iterrows():
    plt.text(row['Quantity_Sold'] + 5, idx, f"{row['Quantity_Sold']}", va='center', fontsize=10)

plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# Get data from previous SQL query
df = _sqldf.toPandas()

# Create combo chart with two y-axes
fig, ax1 = plt.subplots(figsize=(12, 6))

# Bar chart for revenue
color1 = '#2ca02c'
ax1.bar(df['product_type'], df['Total_Revenue'], color=color1, alpha=0.6, label='Revenue')
ax1.set_xlabel('Product Type', fontsize=12)
ax1.set_ylabel('Total Revenue ($)', color=color1, fontsize=12)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.tick_params(axis='x', rotation=45)

# Line chart for quantity on secondary axis
ax2 = ax1.twinx()
color2 = '#ff7f0e'
ax2.plot(df['product_type'], df['Quantity_Sold'], color=color2, marker='o', linewidth=2, markersize=8, label='Quantity')
ax2.set_ylabel('Quantity Sold', color=color2, fontsize=12)
ax2.tick_params(axis='y', labelcolor=color2)

plt.title('Top 5 Revenue Drivers: Revenue vs Volume', fontsize=14, fontweight='bold')

# Add legends
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

# Calculate and display average revenue per transaction
df['Avg_Revenue_Per_Transaction'] = df['Total_Revenue'] / df['Quantity_Sold']
print("\nAverage Revenue per Transaction:")
for idx, row in df.iterrows():
    print(f"{row['product_type']}: ${row['Avg_Revenue_Per_Transaction']:.2f}")

Which days of the week tend to be busiest, and why do you think that's the case?

### Overall Pattern

Friday is the busiest day with 21,701 transactions, followed closely by Thursday (21,654) and Monday (21,643)
Saturday is the slowest day with only 20,510 transactions - notably lower than other days
The difference between busiest and slowest is about 1,200 transactions (~5.8%)
Why Friday-Thursday-Monday are busiest:

Weekday work culture: People need their coffee fix during the work week, especially starting Monday and ending the week strong on Thursday/Friday
Friday social factor: Could be more group orders or people treating themselves before the weekend
Monday momentum: People returning to work need that caffeine boost
Saturday dip: People sleep in on weekends, brew coffee at home, or have different routines
Month-by-Month Variation (Cell 8): Your second query reveals that the busiest day varies by month, which is fascinating:

June had the highest single-day volume (Friday with 5,960 transactions)
April's Sunday was unusually busy (4,279) - could indicate special events or seasonal patterns
No consistent pattern - different days win each month, suggesting external factors (weather, holidays, local events) influence traffic
What this tells you about the business:

Staffing should prioritize Thu-Fri-Mon coverage
Saturday might be an opportunity for promotions to drive traffic
The monthly variation suggests you should analyze specific dates for holidays/events that drove Sunday/Tuesday spikes

In [0]:
%sql
SELECT 
    WEEKDAY(transaction_date) + 1 AS NUM_of_Week,
    DAYNAME(transaction_date) AS Day_of_week,
    COUNT(*) AS Number_of_Transactions,
    ROUND(
        COUNT(*) / SUM(COUNT(*)) OVER() * 100,
        2
    ) AS Percentage,
    RANK() OVER (ORDER BY COUNT(*) DESC) AS RANK
FROM coffee_shop_sales
GROUP BY 
    WEEKDAY(transaction_date),
    DAYNAME(transaction_date)
ORDER BY NUM_of_Week;




Databricks visualization. Run in Databricks to view.

In [0]:
%sql
WITH aggregate_transcations AS(
SELECT 
MONTH(transaction_date) AS MonthNum
,MONTHNAME(transaction_date) AS Month_of_year
,dayofweek(transaction_date) AS Day_of_week_num
,DAYNAME(transaction_date) AS Day_of_week
,COUNT(*) AS Number_of_Transactions
FROM coffee_shop_sales
GROUP BY 
    MonthNum
    ,Month_of_year
    ,Day_of_week_num
    ,Day_of_week
),
RANKING AS (
SELECT 
MonthNum
,Month_of_year
,Day_of_week_num
,Day_of_week
,Number_of_Transactions
,Rank() OVER(PARTITION BY Month_of_year ORDER BY Number_of_Transactions DESC) AS RNK
FROM aggregate_transcations
) 
SELECT 
    *
    
FROM RANKING
WHERE RNK <= 2
ORDER BY MonthNum;

Databricks visualization. Run in Databricks to view.

What times of day tend to be most popular?

### Time of Day Patterns:

Clear Morning Rush (7-10 AM): Peak period with 10 AM hitting 18,545 transactions - the single busiest hour.
Early Morning Surge: 7-9 AM is extremely strong (13K-18K each hour) - classic coffee shop commute pattern.

### Business Insights:

Staffing priority: 7-10 AM is critical - need maximum coverage.
Prep timing: Highest inventory should be ready by 7 AM.

In [0]:
%sql
SELECT 
      HOUR(RIGHT(TRY_CAST(transaction_time AS timestamp),9)) AS Hour
      ,CASE 
        WHEN HOUR(RIGHT(TRY_CAST(transaction_time AS timestamp),9)) = 0 THEN '12 AM'
        WHEN HOUR(RIGHT(TRY_CAST(transaction_time AS timestamp),9)) BETWEEN 1 AND 11 THEN CONCAT(HOUR(RIGHT(TRY_CAST(transaction_time AS timestamp),9)), ' AM')
        WHEN HOUR(RIGHT(TRY_CAST(transaction_time AS timestamp),9)) = 12 THEN '12 PM'
        WHEN HOUR(RIGHT(TRY_CAST(transaction_time AS timestamp),9)) BETWEEN 13 AND 23 THEN CONCAT(HOUR(RIGHT(TRY_CAST(transaction_time AS timestamp),9)) - 12, ' PM')
        ELSE NULL
      END AS Time_of_Day 
    ,COUNT(*) AS Total_Transactions
FROM coffee_shop_sales
GROUP BY Time_of_Day, Hour
ORDER BY Hour ASC;


Databricks visualization. Run in Databricks to view.

Which products are sold most and least often? Which drive the most revenue for the business?

### Product Performance Analysis:

### Most Sold:
Brewed Chai tea (17,183) - volume leader
Gourmet brewed coffee (16,912)
Barista Espresso (16,403)

### Least Sold:
Green beans (134) - 128x less than top seller
Green tea (159)
House blend Beans (183)

### Revenue Drivers:
Barista Espresso - $91,406 (despite being #3 in volume)
Brewed Chai tea - $77,082
Hot chocolate - $72,416


**Key Insight: Value Arbitrage**
Barista Espresso generates 19% more revenue than Chai tea despite 5% fewer transactions.

Espresso avg: $5.57/transaction
Chai avg: $4.49/transaction

24% price premium
Hot chocolate is the margin champion: $6.31/transaction with half the volume of leaders

Strategic Actions:
Discontinue/Clearance:

In [0]:
%sql

-- Top 5 Products Sold the Most 
SELECT 
product_type
,COUNT(product_id) AS Quantity_Sold 
FROM coffee_shop_sales
GROUP BY 
        product_type
ORDER BY Quantity_Sold DESC
LIMIT 5;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Top 5 products with least sales 
SELECT 
product_type
,COUNT(product_id) AS Quantity_Sold 
FROM coffee_shop_sales
GROUP BY 
        product_type
ORDER BY Quantity_Sold ASC
LIMIT 5;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Top 5 products that drive the most revenue for the business 
SELECT 
 product_type
,COUNT(product_id) AS Quantity_Sold 
,ROUND(SUM(transaction_qty * unit_price),0) AS Total_Revenue
FROM coffee_shop_sales
GROUP BY 
        product_type
ORDER BY Total_Revenue DESC
LIMIT 5;

Databricks visualization. Run in Databricks to view.